In [2]:
import torch, cs336_basics;
print(torch.__version__, cs336_basics.__file__)

2.11.0 /Users/pjurczen/repo/cs336-assignment2-systems/cs336-basics-own/cs336_basics/__init__.py


In [7]:
import pandas as pd

kern = pd.read_csv("nsys-reports/nsys/small_full_cuda_gpu_kern_sum.csv")
kern.head(20)

,Time (%),Total Time (ns),Instances,Avg (ns),Med (ns),Min (ns),Max (ns),StdDev (ns),Name
0,12.4,121037614,9240,13099.3,7840.0,2080,150977,20363.3,"void at::native::vectorized_elementwise_kernel<(int)4, at::native::CUDAFunctor_add<float>, std::array<char *, (unsigned long)3>>(int, T2, T3)"
1,9.4,91813499,2565,35794.7,43488.0,6433,95457,29272.5,"void at::native::vectorized_elementwise_kernel<(int)4, at::native::BinaryFunctor<float, float, float, at::native::binary_internal::MulFunctor<float>>, std::array<char *, (unsigned long)3>>(int, T2, T3)"
2,7.4,72122271,2220,32487.5,11552.0,8641,82657,32137.2,"void at::native::elementwise_kernel<(int)128, (int)2, void at::native::gpu_kernel_impl_nocast<at::native::BinaryFunctor<float, float, float, at::native::binary_internal::DivFunctor<float>>>(at::TensorIteratorBase &, const T1 &)::[lambda(int) (instance 1)]>(int, T3)"
3,6.9,67058874,8702,7706.1,6704.0,1920,42721,6116.2,"void at::native::vectorized_elementwise_kernel<(int)4, at::native::AUnaryFunctor<float, float, float, at::native::binary_internal::MulFunctor<float>>, std::array<char *, (unsigned long)2>>(int, T2, T3)"
4,5.6,54471208,3975,13703.4,10592.0,6176,105089,9821.7,"void at::native::elementwise_kernel<(int)128, (int)2, void at::native::gpu_kernel_impl_nocast<at::native::direct_copy_kernel_cuda(at::TensorIteratorBase &)::[lambda() (instance 3)]::operator ()() const::[lambda() (instance 7)]::operator ()() const::[lambda(float) (instance 1)]>(at::TensorIterato..."
5,5.3,52122995,735,70915.6,85856.0,27328,268706,36924.9,void cutlass::Kernel2<cutlass_80_tensorop_s1688gemm_128x128_32x3_nn_align4>(T1::Params)
6,5.2,50510797,540,93538.5,98465.0,82112,99681,7319.5,void cutlass::Kernel2<cutlass_80_tensorop_s1688gemm_256x128_32x3_nt_align4>(T1::Params)
7,4.7,46363824,540,85858.9,93185.0,70112,94369,10711.6,void cutlass::Kernel2<cutlass_80_tensorop_s1688gemm_128x128_16x5_tn_align4>(T1::Params)
8,4.5,44283427,3825,11577.4,8544.0,5632,106497,13860.0,"void at::native::elementwise_kernel<(int)128, (int)2, void at::native::gpu_kernel_impl_nocast<at::native::BinaryFunctor<float, float, float, at::native::binary_internal::MulFunctor<float>>>(at::TensorIteratorBase &, const T1 &)::[lambda(int) (instance 1)]>(int, T3)"
9,3.2,31340853,945,33164.9,48129.0,6880,67809,21116.3,"void at::native::reduce_kernel<(int)512, (int)1, at::native::ReduceOp<float, at::native::func_wrapper_t<float, at::native::sum_functor<float, float, float>::operator ()(at::TensorIterator &)::[lambda(float, float) (instance 1)]>, unsigned int, float, (int)4, (int)4>>(T3)"


In [11]:
s = torch.tensor(0,dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float32)
print(s)

s = torch.tensor(0,dtype=torch.float16)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float16)
print(s)

s = torch.tensor(0,dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float16)
print(s)

s = torch.tensor(0,dtype=torch.float32)
for i in range(1000):
    x = torch.tensor(0.01,dtype=torch.float16)
    s += x.type(torch.float32)
print(s)

tensor(10.0001)
tensor(9.9531, dtype=torch.float16)
tensor(10.0021)
tensor(10.0021)


In [14]:
from torch import nn

class ToyModel(nn.Module):
    
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 10, bias=False)
        self.ln = nn.LayerNorm(10)
        self.fc2 = nn.Linear(10, out_features, bias=False)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.ln(x)
        x = self.fc2(x)
        return x

In [18]:
import torch
from torch import nn

x = torch.randn((4, 512, 2560), requires_grad=True)

class RMSNorm(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        eps: float = 1e-5,
        device=None,
    ):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size, device=device))
        self.eps = eps
        
    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        x = x * rms
        return self.weight * x
        
def pack_hook(t):
    shape, dtype, grad_fn = t.shape, t.dtype, t.grad_fn
    print(f"Saving residual: {shape=}, {dtype=}, {grad_fn=}")
    return t
    
def unpack_hook(t):
    shape, dtype, grad_fn = t.shape, t.dtype, t.grad_fn
    print(f"Loading residual: {shape=}, {dtype=}, {grad_fn=}")
    return t
    
ln = torch.compile(RMSNorm(x.shape[-1]))

with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = ln(x)
    y.sum().backward()

Saving residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([2560]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 512, 1]), dtype=torch.float32, grad_fn=None
Loading residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=None
Loading residual: shape=torch.Size([2560]), dtype=torch.float32, grad_fn=None
Loading residual: shape=torch.Size([4, 512, 1]), dtype=torch.float32, grad_fn=None


In [23]:
import torch
from cs336_basics.transformer import TransformerBlock
from cs336_basics.rope import RotaryPositionalEmbedding

# num_layers for this model is 32
d_model, d_ff, num_heads, context_length = 2560, 10240, 16, 2048

block = TransformerBlock(d_model=d_model, d_ff=d_ff, num_heads=num_heads, theta=10000, max_seq_len=context_length)

# Fuse as much torch.compile will allow
block = torch.compile(block, fullgraph=True)

x = torch.randn((4, context_length, d_model), requires_grad=True)

# Now logs the number of bytes saved
total_size_bytes = 0
def pack_hook(t):
    if isinstance(t, torch.nn.Parameter): # Skip logging parameters to avoid double counting
        return t
    global total_size_bytes
    shape, dtype, grad_fn = t.shape, t.dtype, t.grad_fn
    total_size_bytes += t.numel() * t.element_size()
    print(f"Saving residual: {shape=}, {dtype=}, {grad_fn=}")
    return t
...
# Run forward pass, saving for backward
with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = block(x)
print(f"Total size of saved tensors in single TransformerBlock: {total_size_bytes / (1024**2):.2f} MiB")

python3(66978) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(66981) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(66982) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(66983) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(66984) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(66985) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(66986) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Saving residual: shape=torch.Size([4, 2048, 2560]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([2048, 80]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([2048, 80]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 2048, 1]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 2048, 2560]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([64, 2048, 2048]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([2048, 2048]), dtype=torch.bool, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 1]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 1]), dtype=torch.int64, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 1]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 2048]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([1, 8192, 2560]), dtype=torch.flo

In [51]:
import torch
from cs336_basics.attention import scaled_dot_product_attention
from cs336_basics.cross_entropy import cross_entropy
import timeit
import statistics

device = "mps"
batch_size = 8
d_models = [16, 32, 64, 128]
seq_lens = [256, 1024, 4096, 8192, 16384]

def synchronize(device: str):
    if device == "mps":
        torch.mps.synchronize()
    elif device == "cuda" or device == "cuda:0":
        torch.cuda.synchronize()

def attention_benchmark(d_model: int, seq_len: int) -> None:
    casual_mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))
    Q = torch.randn((batch_size, seq_len, d_model), device=device, requires_grad=True)
    K = torch.randn((batch_size, seq_len, d_model), device=device, requires_grad=True)
    V = torch.randn((batch_size, seq_len, d_model), device=device, requires_grad=True)
    grad_out = torch.randn((batch_size, seq_len, d_model), device=device)
    backward_timing = []
    forward_timing = []
    backward_timing_compiled = []
    forward_timing_compiled = []

    compiled_attention = torch.compile(scaled_dot_product_attention)

    print(f"Benchmarking d_model: {d_model}, seq_len: {seq_len}")
    
    # Warmup steps
    for _ in range(2):
        logits: torch.Tensor = scaled_dot_product_attention(Q, K, V, casual_mask)
        logits_c: torch.Tensor = compiled_attention(Q, K, V, casual_mask)
        logits.backward(grad_out)
        logits_c.backward(grad_out)
    synchronize(device)

    for i in range(100):
        t0 = timeit.default_timer()
        logits: torch.Tensor = scaled_dot_product_attention(Q, K, V, casual_mask)
        synchronize(device)
        t1 = timeit.default_timer()
        
        logits.backward(grad_out)
        synchronize(device)
        t2 = timeit.default_timer()

        t0c = timeit.default_timer()
        logits_c: torch.Tensor = compiled_attention(Q, K, V, casual_mask)
        synchronize(device)
        t1c = timeit.default_timer()

        logits_c.backward(grad_out)
        synchronize(device)
        t2c = timeit.default_timer()
        
        forward_timing.append(t1 - t0)
        backward_timing.append(t2 - t1)
        forward_timing_compiled.append(t1c - t0c)
        backward_timing_compiled.append(t2c - t1c)
    
    print(f"Forward pass: {statistics.mean(forward_timing) * 1000:.2f} ± {statistics.stdev(forward_timing) * 1000:.2f} ms")
    print(f"Backward pass: {statistics.mean(backward_timing) * 1000:.2f} ± {statistics.stdev(backward_timing) * 1000:.2f} ms")
    print(f"Forward pass compiled: {statistics.mean(forward_timing_compiled) * 1000:.2f} ± {statistics.stdev(forward_timing_compiled) * 1000:.2f} ms")
    print(f"Backward pass compiled: {statistics.mean(backward_timing_compiled) * 1000:.2f} ± {statistics.stdev(backward_timing_compiled) * 1000:.2f} ms")


for d_model in d_models[:1]:
    for seq_len in seq_lens[:1]:
        attention_benchmark(d_model, seq_len)

Benchmarking d_model: 16, seq_len: 256
Forward pass: 0.66 ± 0.26 ms
Backward pass: 0.98 ± 0.42 ms
Forward pass compiled: 0.39 ± 0.08 ms
Backward pass compiled: 0.48 ± 0.10 ms


In [56]:
x = torch.arange(12).reshape(3, 4)
print(x.stride())      # (4, 1)
print(x.t().stride())  # (1, 4)

(4, 1)
(1, 4)


In [183]:
import torch
from torch.autograd.function import FunctionCtx
from cs336_basics.attention import scaled_dot_product_attention
from einops import einsum
import math
from jaxtyping import Bool, Float

device = "cuda"

def logsumexp(Q: torch.Tensor, K: torch.Tensor, mask: Bool[torch.Tensor, " ... queries keys"] | None = None) -> torch.Tensor:
    d_k = Q.shape[-1]
    qk = einsum(Q, K, "... queries d_k, ... keys d_k -> ... queries keys") / math.sqrt(d_k)
    if mask is not None:
        qk = qk.masked_fill(~mask, float("-Inf"))
    x_max = qk.max(dim=-1, keepdim=True).values
    x_norm = qk.subtract(x_max)
    x_norm_exp = x_norm.exp()
    result = (x_max + x_norm_exp.sum(dim=-1, keepdim=True).log()).squeeze(dim=-1)
    assert torch.allclose(result, torch.logsumexp(qk, dim=-1))
    return result

class FlashAttention2Func(torch.autograd.Function):

    @staticmethod
    def forward(ctx: FunctionCtx, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, is_causal: bool=False) -> torch.Tensor:
        Q_TILE_SIZE, K_TILE_SIZE = 16, 16
        device, dtype = Q.device, Q.dtype
        q_tiles_count = math.ceil(Q.shape[-2] / Q_TILE_SIZE) # (2)
        k_tiles_count = math.ceil(K.shape[-2] / K_TILE_SIZE) # (3)
        d_k = Q.shape[-1]
        O = torch.zeros_like(Q)
        L = torch.zeros(Q.shape[:-1], device=device, dtype=dtype)
        
        for i in range(q_tiles_count): # (4)
            Q_i = Q[..., i * Q_TILE_SIZE : (i + 1) * Q_TILE_SIZE, :] # (5)
            O_i = torch.zeros_like(Q_i) # (6)
            l_i = torch.zeros(Q_i.shape[:-1], device=device, dtype=dtype) # (6)
            m_i = torch.full(Q_i.shape[:-1], float('-inf'), device=device, dtype=dtype) # (6)
            for j in range(k_tiles_count): # (7)
                K_j = K[..., j * K_TILE_SIZE : (j + 1) * K_TILE_SIZE, :] # (8)
                V_j = V[..., j * K_TILE_SIZE : (j + 1) * K_TILE_SIZE, :] # (8)
                S_ij = einsum(Q_i, K_j, "... q d, ... k d -> ... q k") / math.sqrt(d_k) # (9)
                rowmax_S_ij = S_ij.max(dim=-1).values # (10)
                m_ij = torch.max(m_i, rowmax_S_ij) # (10)
                P_ij = torch.exp(S_ij - m_ij[..., None]) # (11)
                rowsum_P_ij = P_ij.sum(dim=-1) # (12)
                l_ij = (torch.exp(m_i - m_ij) * l_i) + rowsum_P_ij # (12)
                Pij_Vj = einsum(P_ij, V_j, "... Bq Bk, ... Bk d -> ... Bq d") # (13)
                O_i = torch.exp(m_i - m_ij)[..., None] * O_i + Pij_Vj # (13)
                m_i = m_ij
                l_i = l_ij
                
            O_i = O_i / l_i[..., None] # (15)
            L_i = m_i + l_i.log() # (16)
            
            O[..., i * Q_TILE_SIZE : (i + 1) * Q_TILE_SIZE, :] = O_i # (17)
            L[..., i * Q_TILE_SIZE : (i + 1) * Q_TILE_SIZE] = L_i # (18)
        
        ctx.save_for_backward(L, Q, K, V, O)
        return O

    @staticmethod
    def backward(ctx, grad_out):
        raise NotImplementedError

batch_size = 4
seq_len = 32
d_k = d_v = 64
Q = torch.randn((batch_size, seq_len, d_k), requires_grad=True, device=device)
K = torch.randn((batch_size, seq_len, d_k), requires_grad=True, device=device)
V = torch.randn((batch_size, seq_len, d_v), requires_grad=True, device=device)

baseline_O = scaled_dot_product_attention(Q, K, V, None)
result = FlashAttention2Func.apply(Q, K, V)
baseline_L = logsumexp(Q, K)
L, Q, K, V, O = result.grad_fn.saved_tensors
print(L.shape)
torch.testing.assert_close(L, baseline_L)
torch.testing.assert_close(O, baseline_O)

AssertionError: Torch not compiled with CUDA enabled

In [203]:
import triton
import triton.language as tl

@triton.jit
def flash_fwd_kernel(
    Q_ptr, K_ptr, V_ptr,
    O_ptr, L_ptr,
    stride_qb, stride_qq, stride_qd,
    stride_kb, stride_kk, stride_kd,
    stride_vb, stride_vk, stride_vd,
    stride_ob, stride_oq, stride_od,
    stride_lb, stride_lq,
    N_QUERIES, N_KEYS,
    scale,
    D: tl.constexpr,
    Q_TILE_SIZE: tl.constexpr,
    K_TILE_SIZE: tl.constexpr,
    is_casual: tl.constexpr,
):
    # Program indices
    query_tile_index = tl.program_id(0)
    batch_index = tl.program_id(1)
    # Offset each pointer with the corresponding batch index
    # multiplied with the batch stride for each tensor
    Q_block_ptr = tl.make_block_ptr(
        Q_ptr + batch_index * stride_qb,
        shape=(N_QUERIES, D),
        strides=(stride_qq, stride_qd),
        offsets=(query_tile_index * Q_TILE_SIZE, 0),
        block_shape=(Q_TILE_SIZE, D),
        order=(1, 0),
    )

    K_block_ptr = tl.make_block_ptr(
        K_ptr + batch_index * stride_kb,
        shape=(N_KEYS, D),
        strides=(stride_kk, stride_kd),
        offsets=(0, 0),
        block_shape=(K_TILE_SIZE, D),
        order=(1, 0),
    )

    V_block_ptr = tl.make_block_ptr(
        V_ptr + batch_index * stride_vb,
        shape=(N_KEYS, D),
        strides=(stride_vk, stride_vd),
        offsets=(0, 0),
        block_shape=(K_TILE_SIZE, D),
        order=(1, 0),
    )

    O_block_ptr = tl.make_block_ptr(
        O_ptr + batch_index * stride_ob,
        shape=(N_QUERIES, D),
        strides=(stride_oq, stride_od),
        offsets=(query_tile_index * Q_TILE_SIZE, 0),
        block_shape=(Q_TILE_SIZE, D),
        order=(1, 0),
    )

    L_block_ptr = tl.make_block_ptr(
        L_ptr + batch_index * stride_lb,
        shape=(N_QUERIES, ),
        strides=(stride_lq, ),
        offsets=(query_tile_index * Q_TILE_SIZE, ),
        block_shape=(Q_TILE_SIZE, ),
        order=(0, ),
    )

    O_i = tl.zeros((Q_TILE_SIZE, D), dtype=tl.float32) # (6)
    l_i = tl.zeros((Q_TILE_SIZE, ), dtype=tl.float32) # (6)
    m_i = tl.full((Q_TILE_SIZE, ), float('-inf'), dtype=tl.float32) # (6)

    Q_i = tl.load(Q_block_ptr) # (Q_TILE_SIZE, D)

    q_pos = query_tile_index * Q_TILE_SIZE + tl.arange(0, Q_TILE_SIZE)
    k_arr = tl.arange(0, K_TILE_SIZE)

    for j in range(tl.cdiv(N_KEYS, K_TILE_SIZE)):
        K_j = tl.load(K_block_ptr) # (K_TILE_SIZE, D)
        V_j = tl.load(V_block_ptr) # (K_TILE_SIZE, D)
        
        k_pos = j * K_TILE_SIZE + k_arr
        mask = k_pos[None, :] > q_pos[:, None]
        
        S_ij = tl.dot(Q_i, tl.trans(K_j)) * scale # (Q_TILE_SIZE, K_TILE_SIZE) (9)
        S_ij= tl.where(mask, S_ij - 1e6, S_ij)
        rowmax_S_ij = tl.max(S_ij, axis=1) # (Q_TILE_SIZE, ) (10)
        m_ij = tl.maximum(m_i, rowmax_S_ij) # (Q_TILE_SIZE, ) (10)
        P_ij = tl.exp(S_ij - tl.expand_dims(m_ij, 1)) # (Q_TILE_SIZE, K_TILE_SIZE) (11)
        rowsum_P_ij = tl.sum(P_ij, axis=-1) # (Q_TILE_SIZE, ) (12)
        l_ij = (tl.exp(m_i - m_ij) * l_i) + rowsum_P_ij # (Q_TILE_SIZE, ) (12)
        O_i = tl.expand_dims(tl.exp(m_i - m_ij), 1) * O_i # (13)
        O_i = tl.dot(P_ij.to(V_j.dtype), V_j, acc=O_i) # (13)

        K_block_ptr = K_block_ptr.advance((K_TILE_SIZE, 0))
        V_block_ptr = V_block_ptr.advance((K_TILE_SIZE, 0))
        m_i = m_ij
        l_i = l_ij

    O_i = O_i / tl.expand_dims(l_i, 1) # (15)
    L_i = m_i + tl.log(l_i) # (16)

    tl.store(O_block_ptr, O_i.to(O_block_ptr.type.element_ty))
    tl.store(L_block_ptr, L_i.to(L_block_ptr.type.element_ty))
    

In [204]:
import math
import torch

device = "cuda"
batch_size = 4
seq_len = 32
d_k = d_v = 64
Q = torch.randn((batch_size, seq_len, d_k), requires_grad=True, device=device, dtype=torch.bfloat16)
K = torch.randn((batch_size, seq_len, d_k), requires_grad=True, device=device, dtype=torch.bfloat16)
V = torch.randn((batch_size, seq_len, d_v), requires_grad=True, device=device, dtype=torch.bfloat16)
Q_TILE_SIZE, K_TILE_SIZE = 16, 32
N_QUERIES, N_KEYS = Q.shape[-2], K.shape[-2]
scale = 1/math.sqrt(d_k)
D = d_k
is_casual=True

Tq = triton.cdiv(N_QUERIES, Q_TILE_SIZE)

O = torch.zeros_like(Q)
L = torch.zeros(Q.shape[:-1], device=Q.device, dtype=Q.dtype)

device, dtype = Q.device, Q.dtype
q_tiles_count = math.ceil(Q.shape[-2] / Q_TILE_SIZE) # (2)
k_tiles_count = math.ceil(K.shape[-2] / K_TILE_SIZE) # (3)

flash_fwd_kernel[(Tq, batch_size)](
    Q, K, V,
    O, L,
    Q.stride(0), Q.stride(1), Q.stride(2),
    K.stride(0), K.stride(1), K.stride(2),
    V.stride(0), V.stride(1), V.stride(2),
    O.stride(0), O.stride(1), O.stride(2),
    L.stride(0), L.stride(1),
    N_QUERIES, N_KEYS,
    scale,
    D=D,
    Q_TILE_SIZE=Q_TILE_SIZE,
    K_TILE_SIZE=K_TILE_SIZE,
    is_casual=is_casual
)

casual_mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))
baseline_O = scaled_dot_product_attention(Q, K, V, casual_mask)
baseline_L = logsumexp(Q, K, casual_mask)

torch.testing.assert_close(L, baseline_L, rtol=1e-2, atol=1e-2)
torch.testing.assert_close(O, baseline_O, rtol=1e-2, atol=1e-2)

In [186]:
torch.arange(0, 4)
q_pos = 1 * 4 + torch.arange(0, 4)
k_pos = 0 * 8 + torch.arange(0, 8)
q_pos[:, None]
k_pos[None, :]
mask = k_pos[None, :] > q_pos[:, None]
mask

tensor([[False, False, False, False, False,  True,  True,  True],
        [False, False, False, False, False, False,  True,  True],
        [False, False, False, False, False, False, False,  True],
        [False, False, False, False, False, False, False, False]])

In [191]:
S = torch.zeros(4, 8)
torch.where(mask, S - 1e6, S)

tensor([[       0.,        0.,        0.,        0.,        0., -1000000.,
         -1000000., -1000000.],
        [       0.,        0.,        0.,        0.,        0.,        0.,
         -1000000., -1000000.],
        [       0.,        0.,        0.,        0.,        0.,        0.,
                0., -1000000.],
        [       0.,        0.,        0.,        0.,        0.,        0.,
                0.,        0.]])

In [206]:
import torch
from cs336_basics.attention import scaled_dot_product_attention
from einops import einsum

device = "mps"
batch_size = 4
seq_len = 32
d_k = d_v = 64
scale = math.sqrt(d_k)

casual_mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))

Q = torch.randn((batch_size, seq_len, d_k), requires_grad=True, device=device)
K = torch.randn((batch_size, seq_len, d_k), requires_grad=True, device=device)
V = torch.randn((batch_size, seq_len, d_v), requires_grad=True, device=device)
O = scaled_dot_product_attention(Q, K, V, casual_mask)
dO = torch.randn((batch_size, seq_len, d_v), device=device)
L = logsumexp(Q, K, casual_mask)

def flash_bwd(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, O: torch.Tensor, dO: torch.Tensor, L: torch.Tensor) -> tuple[torch.Tensor, ...]:
    D = torch.sum(O * dO, dim=-1)
    S = einsum(Q, K, "... q d, ... k d -> ... q k") / scale
    S = S.masked_fill(~casual_mask, float("-Inf"))
    P = torch.exp(S - L[... , None])
    dV = einsum(P, dO, "... q k, ... q d -> ... k d")
    dP = einsum(dO, V, "... q d, ... k d -> ... q k")
    dS = P * (dP - D[..., None])
    dQ = einsum(dS, K, "... q k, ... k d -> ... q d") / scale
    dK = einsum(dS, Q, "... q k, ... q d -> ... k d") / scale
    return dQ, dK, dV

flash_bwd(Q, K, V, O, dO, L)

(tensor([[[-1.6652e-07,  6.2684e-08,  8.0941e-08,  ..., -6.9330e-08,
           -1.0269e-07,  1.8776e-07],
          [-8.7415e-02,  2.4630e-02,  9.0619e-02,  ..., -1.1809e-02,
           -9.7038e-02,  4.7293e-02],
          [ 1.7380e-01,  2.5787e-01,  1.7496e-01,  ...,  4.1192e-01,
           -2.3100e-01, -3.3817e-01],
          ...,
          [ 3.5364e-01,  3.7007e-02, -4.1121e-02,  ...,  1.6050e-01,
            1.2412e-01, -1.5056e-01],
          [-5.1404e-02, -4.0449e-01,  1.2771e-01,  ..., -8.1588e-02,
            9.1112e-02, -3.2463e-02],
          [-1.3629e-01,  5.0897e-01,  2.3455e-01,  ..., -1.5926e-01,
            3.7183e-01, -6.0098e-02]],
 
         [[ 5.4520e-08,  1.0600e-07, -3.0708e-08,  ...,  2.4301e-08,
           -2.1562e-08, -3.4149e-08],
          [ 1.2452e-02, -4.3203e-01, -2.4358e-02,  ...,  4.5571e-02,
            1.4146e-01, -1.2283e-01],
          [ 2.6697e-01,  1.9758e-01, -1.6616e-01,  ...,  2.3542e-01,
            2.9965e-01, -1.8118e-01],
          ...,
    

In [26]:
from cs336_systems.flash_attention_triton import FlashAttention2TritonFunc
from cs336_basics.attention import scaled_dot_product_attention
import triton
import math
import torch

device = "cuda"
batch_size = 1
seq_len = 32
d_k = d_v = 64

Q = torch.randn((batch_size, seq_len, d_k), requires_grad=True, device=device)
K = torch.randn((batch_size, seq_len, d_k), requires_grad=True, device=device)
V = torch.randn((batch_size, seq_len, d_v), requires_grad=True, device=device)
grad_out = torch.randn((batch_size, seq_len, d_k), device=device)

flash = FlashAttention2TritonFunc.apply

def forward_only():
    flash(Q, K, V, True)

O = flash(Q, K, V, True)
def backward_only():
    O.backward(grad_out, retain_graph=True)

def forward_backward():
    O = flash(Q, K, V, True)
    O.backward(grad_out)

fwd_triton = triton.testing.do_bench(forward_only, warmup=25, rep=100)
bwd_triton = triton.testing.do_bench(backward_only, warmup=25, rep=100, grad_to_none=[Q, K, V])
fwd_bwd_triton = triton.testing.do_bench(forward_backward, warmup=25, rep=100, grad_to_none=[Q, K, V])

casual_mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))

def forward_naive():
    scaled_dot_product_attention(Q, K, V, casual_mask)

O = scaled_dot_product_attention(Q, K, V, casual_mask)
def backward_naive():
    O.backward(grad_out, retain_graph=True)

def forward_backward_naive():
    O = scaled_dot_product_attention(Q, K, V, casual_mask)
    O.backward(grad_out)

fwd_naive = triton.testing.do_bench(forward_naive, warmup=25, rep=100)
bwd_naive = triton.testing.do_bench(backward_naive, warmup=25, rep=100, grad_to_none=[Q, K, V])
fwd_bwd_naive = triton.testing.do_bench(forward_backward_naive, warmup=25, rep=100, grad_to_none=[Q, K, V])

